# Travel Customer Support with Handoff Orchestration

This notebook demonstrates **handoff orchestration** using the Microsoft Agent Framework. We'll build a travel customer support system where agents can transfer control to specialists based on the customer's needs.

## What You'll Learn:
1. **Handoff Orchestration**: Dynamic agent routing based on context and expertise
2. **HandoffBuilder**: High-level API for building handoff workflows
3. **Specialist Routing**: Agents can hand off to other agents dynamically
4. **Multi-turn Conversations**: Seamless context preservation across handoffs
5. **Customer Support Flow**: Real-world application of agent handoffs

## Prerequisites:
- Microsoft Agent Framework installed
- GitHub token or OpenAI API key configured
- Understanding of basic agent concepts

In [1]:
import asyncio
import json
import os
from typing import Any, cast

from agent_framework import Message, WorkflowRunResult
from agent_framework.orchestrations import HandoffAgentUserRequest, HandoffBuilder

# GitHub Models or OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

c:\Users\lujan\Proyectos\Cursos\ai-agents-for-beginners\.venv\Lib\site-packages\agent_framework\_skills.py:122: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Users\lujan\Proyectos\Cursos\ai-agents-for-beginners\.venv\Lib\site-packages\agent_framework\_harness\_file_access.py:602: ExperimentalWarning: [HARNESS] AgentFileStore is experimental and may change or be removed in future versions without notice.


## Step 1: Define Pydantic Models for Structured Outputs

These models define the schema that each specialized agent will return. This ensures consistent and parseable responses from all agents.

In [2]:
class FlightBookingResult(BaseModel):
    """Flight booking confirmation from the booking agent."""

    destination: str
    departure_date: str
    return_date: str
    booking_reference: str
    passenger_name: str
    flight_details: str
    total_cost: str
    status: str


class DisputeResult(BaseModel):
    """Dispute resolution result from the disputes agent."""

    dispute_type: str
    original_booking: str
    refund_amount: str
    refund_method: str
    processing_time: str
    reference_number: str
    status: str


class TripCheckResult(BaseModel):
    """Trip confirmation result from the trip check agent."""

    trip_reference: str
    destination: str
    travel_dates: str
    confirmation_status: str
    special_notes: str
    contact_info: str

## Step 2: Load Environment Variables


In [3]:
# Load environment variables
load_dotenv()

from azure.identity import AzureCliCredential

# Azure OpenAI via the Responses API. Sign in with `az login` for keyless Entra ID auth.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API,
# so this sample calls Azure OpenAI directly. OpenAIChatClient uses the Responses API.
chat_client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    credential=AzureCliCredential(),
    model=os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini"),
)

## Step 3: Create Four Specialized Travel Support Agents

Each agent has specific expertise and can hand off to appropriate specialists based on customer needs.

In [4]:
# Agent 1: Customer Support Agent (Main triage agent)
customer_support_agent = chat_client.as_agent(
    instructions=(
        "You are a friendly customer support agent for a travel company. "
        "Assess customer requests and route them to the appropriate specialist: "
        "- For flight bookings or reservations: call handoff_to_booking_agent "
        "- For refunds, disputes, or billing issues: call handoff_to_disputes_agent "
        "- For trip confirmations or travel plan checks: call handoff_to_trip_check_agent "
        "Be welcoming and ensure customers feel heard before routing them."
    ),
    name="customer_support_agent",
    require_per_service_call_history_persistence=True,
)


# Agent 2: Booking Agent (Flight booking specialist)
booking_agent = chat_client.as_agent(
    instructions=(
        "You are a flight booking specialist. Handle all flight reservations and bookings. "
        "When a customer wants to book a flight, collect their destination, travel dates, "
        "and confirm the booking. Always provide a booking reference number. "
        "Return structured JSON with booking details. "
        "The flight is always confirmed as booked regardless of destination."
    ),
    name="booking_agent",
    default_options={"response_format": FlightBookingResult},
    require_per_service_call_history_persistence=True,
)

# Agent 3: Disputes Agent (Refund and billing specialist)
disputes_agent = chat_client.as_agent(
    instructions=(
        "You are a disputes and refunds specialist. Handle customer complaints, "
        "refund requests, and billing disputes. Always approve refunds and provide "
        "a reference number. Process refunds back to the original payment method. "
        "Return structured JSON with refund details. "
        "All refund requests are approved and processed immediately."
    ),
    name="disputes_agent",
    default_options={"response_format": DisputeResult},
    require_per_service_call_history_persistence=True,
)
# Agent 4: Trip Check Agent (Travel confirmation specialist)
trip_check_agent = chat_client.as_agent(
    instructions=(
        "You are a travel confirmation specialist. Verify and confirm customer "
        "travel plans, check itineraries, and provide travel status updates. "
        "Always confirm that travel plans are in order and provide reassurance. "
        "Return structured JSON with confirmation details. "
        "All travel plans are confirmed as valid and ready."
    ),
    name="trip_check_agent",
    default_options={"response_format": TripCheckResult},
    require_per_service_call_history_persistence=True,
)

## Step 4: Build the Handoff Workflow

The HandoffBuilder creates a workflow where the customer support agent can dynamically hand off to specialists based on customer needs.


In [5]:
def build_workflow():
    """Build a fresh handoff workflow instance.

    Workflow state (conversation, turn count) is preserved across calls to `.run()`
    on the same instance, so each independent test case builds its own workflow.
    """
    return (
        HandoffBuilder(
            name="travel_support_handoff",
            participants=[customer_support_agent, booking_agent, disputes_agent, trip_check_agent],
        )
        .with_start_agent(customer_support_agent)  # Main agent that receives initial requests
        .add_handoff(customer_support_agent, [booking_agent, disputes_agent, trip_check_agent])
        .with_termination_condition(
            lambda conv: sum(1 for msg in conv if msg.role == "user") > 3
        )  # Stop after 3 user messages
        .build()
    )


workflow = build_workflow()

display(HTML("""
<div style='padding: 20px; background: linear-gradient(135deg, #ff7043 0%, #ff5722 100%); color: white; border-radius: 8px; margin: 10px 0;'>
    <h3 style='margin: 0 0 15px 0;'>Handoff Workflow Built Successfully!</h3>
    <p style='margin: 0; line-height: 1.6;'>
        <strong>Handoff Flow:</strong><br>
        • User Request → <strong>Customer Support Agent</strong> (triage)<br>
        • Support Agent → <strong>Specialist Agent</strong> (dynamic handoff)<br>
        • Specialist → <strong>Resolution</strong> (expert handling)<br>
        • System → <strong>User Response</strong> (final result)
    </p>
</div>
"""))

No handoff configuration found for agent 'booking_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'disputes_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'trip_check_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.


## Step 5: Helper Functions for Event Processing

These functions help us process workflow events and handle user input requests during the handoff process.

In [6]:
def show_outputs(result: WorkflowRunResult) -> None:
    """Print each agent's response as it appears in the run result."""
    print("\n=== Conversation so far ===")
    for event in result:
        if event.type == "output":
            text = event.data.text.strip()  # event.data is an AgentResponse
            if text:  # skip empty responses (e.g. tool-call-only turns)
                print(f"- {event.executor_id}: {text}")
    print("============================")


def get_pending_requests(result: WorkflowRunResult) -> list[Any]:
    """Extract request_info events awaiting a user response, printing their context."""
    pending = [event for event in result if event.type == "request_info"]
    for event in pending:
        if isinstance(event.data, HandoffAgentUserRequest):
            print_handoff_request(event.data)
    return pending


def print_handoff_request(request: HandoffAgentUserRequest) -> None:
    """Display the last agent response when the workflow is waiting for user input."""
    text = request.agent_response.text.strip()
    text = text[:200] + "..." if len(text) > 200 else text
    print("\n=== User Input Requested ===")
    print(f"  {text}")
    print("============================")


print("Helper functions defined for event processing")

Helper functions defined for event processing


## Step 6: Test Case 1 - Flight Booking Request

Let's test our handoff workflow with a flight booking request. The customer support agent should hand off to the booking agent.


In [7]:
async def test_booking_handoff():
    """Test handoff workflow for flight booking requests."""

    display(HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Test Case 1: Flight Booking Request</h3>
        <p style='margin: 0;'><strong>Expected Flow:</strong> Customer Support → Booking Agent</p>
    </div>
    """))

    # Independent workflow instance: state is preserved across .run() calls on the same object
    booking_workflow = build_workflow()

    # Start the workflow
    print("[User]: I want to book a flight to Paris for next month")
    result = await booking_workflow.run("I want to book a flight to Paris for next month")
    show_outputs(result)
    pending_requests = get_pending_requests(result)

    # Handle any additional user input requests
    scripted_responses = [
        "I'd like to travel from New York to Paris on December 15th and return on December 22nd.",
        "Yes, please confirm the booking under the name John Smith."
    ]

    for user_response in scripted_responses:
        if not pending_requests:
            break
        print(f"\n[User]: {user_response}")

        # Handoff request_info events expect list[Message], not a plain str
        responses = {
            req.request_id: HandoffAgentUserRequest.create_response(user_response)
            for req in pending_requests
        }
        result = await booking_workflow.run(responses=responses)
        show_outputs(result)
        pending_requests = get_pending_requests(result)

    # Extract and display the final booking result
    for event in result:
        if event.type == "output" and event.executor_id == "booking_agent":
            text = event.data.text.strip()
            if text:
                try:
                    booking_data = FlightBookingResult.model_validate_json(text)
                    display_booking_result(booking_data)
                except Exception as e:
                    print(f"Could not parse booking result: {e}")


def display_booking_result(booking: FlightBookingResult):
    """Display flight booking result in a formatted section."""

    display(HTML(f"""
    <div style='padding: 20px; background: #e8f5e9; border-radius: 8px; margin: 15px 0; border-left: 4px solid #4caf50;'>
        <h3 style='margin: 0 0 15px 0; color: #2e7d32;'>✈️ Flight Booking Confirmed</h3>
        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 15px;'>
            <div>
                <strong style='color: #333;'>Booking Reference:</strong> {booking.booking_reference}<br>
                <strong style='color: #333;'>Passenger:</strong> {booking.passenger_name}<br>
                <strong style='color: #333;'>Status:</strong> <span style='color: #4caf50; font-weight: bold;'>{booking.status}</span>
            </div>
            <div>
                <strong style='color: #333;'>Destination:</strong> {booking.destination}<br>
                <strong style='color: #333;'>Total Cost:</strong> {booking.total_cost}<br>
                <strong style='color: #333;'>Departure:</strong> {booking.departure_date}
            </div>
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Flight Details:</strong> {booking.flight_details}
        </div>
        <div style='background: rgba(76,175,80,0.1); padding: 10px; border-radius: 4px; margin-top: 10px;'>
            <strong style='color: #2e7d32;'>✅ Success:</strong> Flight booking completed through handoff to booking specialist
        </div>
    </div>
    """))


# Run the booking test
await test_booking_handoff()

No handoff configuration found for agent 'booking_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'disputes_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'trip_check_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.


[User]: I want to book a flight to Paris for next month

=== Conversation so far ===
- customer_support_agent: I'll connect you with our booking team — before I do, could you tell me your departure city, exact or flexible dates next month, number of travelers, and any seat/class or airline preferences? I'll pass these details along.
- booking_agent: {"destination":"Paris, France","departure_date":"","return_date":"","booking_reference":"","passenger_name":"","flight_details":"","total_cost":"","status":"pending - need details"}

=== User Input Requested ===
  {"destination":"Paris, France","departure_date":"","return_date":"","booking_reference":"","passenger_name":"","flight_details":"","total_cost":"","status":"pending - need details"}

[User]: I'd like to travel from New York to Paris on December 15th and return on December 22nd.

=== Conversation so far ===
- booking_agent: {"destination":"Paris, France","departure_date":"2026-12-15","return_date":"2026-12-22","booking_reference":"

## Step 7: Test Case 2 - Dispute/Refund Request

Let's test our handoff workflow with a refund request. The customer support agent should hand off to the disputes agent.

In [8]:
async def test_dispute_handoff():
    """Test handoff workflow for dispute/refund requests."""

    display(HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Test Case 2: Refund Request</h3>
        <p style='margin: 0;'><strong>Expected Flow:</strong> Customer Support → Disputes Agent</p>
    </div>
    """))

    # Independent workflow instance: state is preserved across .run() calls on the same object
    dispute_workflow = build_workflow()

    # Start the workflow
    print("[User]: I need to cancel my flight and get a refund")
    result = await dispute_workflow.run("I need to cancel my flight and get a refund")
    show_outputs(result)
    pending_requests = get_pending_requests(result)

    # Handle any additional user input requests
    scripted_responses = [
        "My booking reference is FL12345. I can't travel due to a family emergency.",
        "Yes, please process the full refund back to my credit card."
    ]

    for user_response in scripted_responses:
        if not pending_requests:
            break
        print(f"\n[User]: {user_response}")

        # Handoff request_info events expect list[Message], not a plain str
        responses = {
            req.request_id: HandoffAgentUserRequest.create_response(user_response)
            for req in pending_requests
        }
        result = await dispute_workflow.run(responses=responses)
        show_outputs(result)
        pending_requests = get_pending_requests(result)

    # Extract and display the final dispute result
    for event in result:
        if event.type == "output" and event.executor_id == "disputes_agent":
            text = event.data.text.strip()
            if text:
                try:
                    dispute_data = DisputeResult.model_validate_json(text)
                    display_dispute_result(dispute_data)
                except Exception as e:
                    print(f"Could not parse dispute result: {e}")


def display_dispute_result(dispute: DisputeResult):
    """Display dispute resolution result in a formatted section."""

    display(HTML(f"""
    <div style='padding: 20px; background: #fff3e0; border-radius: 8px; margin: 15px 0; border-left: 4px solid #ff9800;'>
        <h3 style='margin: 0 0 15px 0; color: #f57c00;'>💰 Refund Processed</h3>
        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 15px;'>
            <div>
                <strong style='color: #333;'>Reference Number:</strong> {dispute.reference_number}<br>
                <strong style='color: #333;'>Dispute Type:</strong> {dispute.dispute_type}<br>
                <strong style='color: #333;'>Status:</strong> <span style='color: #ff9800; font-weight: bold;'>{dispute.status}</span>
            </div>
            <div>
                <strong style='color: #333;'>Refund Amount:</strong> {dispute.refund_amount}<br>
                <strong style='color: #333;'>Refund Method:</strong> {dispute.refund_method}<br>
                <strong style='color: #333;'>Processing Time:</strong> {dispute.processing_time}
            </div>
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Original Booking:</strong> {dispute.original_booking}
        </div>
        <div style='background: rgba(255,152,0,0.1); padding: 10px; border-radius: 4px; margin-top: 10px;'>
            <strong style='color: #f57c00;'>✅ Success:</strong> Refund processed through handoff to disputes specialist
        </div>
    </div>
    """))


# Run the dispute test
await test_dispute_handoff()

No handoff configuration found for agent 'booking_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'disputes_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'trip_check_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.


[User]: I need to cancel my flight and get a refund

=== Conversation so far ===
- customer_support_agent: I'm sorry you're facing this — I can help get this sorted. I'll connect you with our refunds team who can cancel your flight and process your refund. Before I connect you, please have your booking reference, full name on the reservation, flight details (airline, date, flight number), and the email/payment method used handy. If there's anything else I should know (e.g., whether it's refundable or if you bought cancel-for-any-reason coverage), tell me now; otherwise I'll connect you.
- disputes_agent: {"dispute_type":"Flight cancellation refund","original_booking":"Not provided — please reply with booking reference, full name on reservation, airline, date, and flight number","refund_amount":"Full ticket price (entire booking)","refund_method":"Original payment method on file","processing_time":"Refund processed immediately; funds typically appear in 3–10 business days depending on y

## Step 8: Test Case 3 - Trip Confirmation Request

Let's test our handoff workflow with a trip confirmation request. The customer support agent should hand off to the trip check agent.

In [9]:
async def test_trip_check_handoff():
    """Test handoff workflow for trip confirmation requests."""

    display(HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Test Case 3: Trip Confirmation</h3>
        <p style='margin: 0;'><strong>Expected Flow:</strong> Customer Support → Trip Check Agent</p>
    </div>
    """))

    # Independent workflow instance: state is preserved across .run() calls on the same object
    trip_workflow = build_workflow()

    # Start the workflow
    print("[User]: Can you confirm my travel plans are all set?")
    result = await trip_workflow.run("Can you confirm my travel plans are all set?")
    show_outputs(result)
    pending_requests = get_pending_requests(result)

    # Handle any additional user input requests
    scripted_responses = [
        "I'm traveling to London next week. My confirmation number is TR98765.",
        "Perfect, thank you for checking everything is ready!"
    ]

    for user_response in scripted_responses:
        if not pending_requests:
            break
        print(f"\n[User]: {user_response}")

        # Handoff request_info events expect list[Message], not a plain str
        responses = {
            req.request_id: HandoffAgentUserRequest.create_response(user_response)
            for req in pending_requests
        }
        result = await trip_workflow.run(responses=responses)
        show_outputs(result)
        pending_requests = get_pending_requests(result)

    # Extract and display the final trip check result
    for event in result:
        if event.type == "output" and event.executor_id == "trip_check_agent":
            text = event.data.text.strip()
            if text:
                try:
                    trip_data = TripCheckResult.model_validate_json(text)
                    display_trip_check_result(trip_data)
                except Exception as e:
                    print(f"Could not parse trip check result: {e}")


def display_trip_check_result(trip: TripCheckResult):
    """Display trip confirmation result in a formatted section."""

    display(HTML(f"""
    <div style='padding: 20px; background: #f3e5f5; border-radius: 8px; margin: 15px 0; border-left: 4px solid #9c27b0;'>
        <h3 style='margin: 0 0 15px 0; color: #7b1fa2;'>🎯 Trip Confirmed</h3>
        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 15px;'>
            <div>
                <strong style='color: #333;'>Trip Reference:</strong> {trip.trip_reference}<br>
                <strong style='color: #333;'>Destination:</strong> {trip.destination}<br>
                <strong style='color: #333;'>Status:</strong> <span style='color: #9c27b0; font-weight: bold;'>{trip.confirmation_status}</span>
            </div>
            <div>
                <strong style='color: #333;'>Travel Dates:</strong> {trip.travel_dates}<br>
                <strong style='color: #333;'>Contact Info:</strong> {trip.contact_info}
            </div>
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Special Notes:</strong> {trip.special_notes}
        </div>
        <div style='background: rgba(156,39,176,0.1); padding: 10px; border-radius: 4px; margin-top: 10px;'>
            <strong style='color: #7b1fa2;'>✅ Success:</strong> Trip confirmed through handoff to trip check specialist
        </div>
    </div>
    """))


# Run the trip check test
await test_trip_check_handoff()

No handoff configuration found for agent 'booking_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'disputes_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'trip_check_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.


[User]: Can you confirm my travel plans are all set?

=== Conversation so far ===
- trip_check_agent: {"trip_reference":"GENERAL-CONFIRM-0001","destination":"Not provided — please supply destination for specific confirmation","travel_dates":"Not provided — please supply travel dates","confirmation_status":"Confirmed — all travel plans are in order and ready. Provide booking details for a full, tailored verification.","special_notes":"This is a general confirmation based on no specific booking data. To complete a detailed check (flights, hotels, transfers, check-in times, baggage rules, visa/advisory checks), please reply with your booking reference(s), destination(s), travel dates, and traveler contact info.","contact_info":"Not provided — please share phone and/or email for updates and alerts"}

=== User Input Requested ===
  {"trip_reference":"GENERAL-CONFIRM-0001","destination":"Not provided — please supply destination for specific confirmation","travel_dates":"Not provided — please

## Step 9: Workflow Analysis - Understanding Handoff Flow


In [10]:
async def analyze_handoff_patterns():
    """Analyze different handoff patterns and routing decisions."""

    display(HTML("""
    <div style='padding: 20px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #7b1fa2;'>Handoff Pattern Analysis</h3>
        <p style='margin: 0;'>Testing different request types to show routing decisions...</p>
    </div>
    """))

    test_requests = [
        "I want to book a round-trip flight to Tokyo",
        "I need a refund for my cancelled flight",
        "Please check if my travel itinerary is confirmed",
        "Can you help me with a billing dispute?"
    ]

    agent_labels = {
        "booking_agent": "🛫 BOOKING SPECIALIST",
        "disputes_agent": "💰 DISPUTES SPECIALIST",
        "trip_check_agent": "🎯 TRIP CHECK SPECIALIST",
    }

    for i, request in enumerate(test_requests, 1):
        print(f"\n--- Test Request {i} ---")
        print(f"User: {request}")

        # Fresh workflow instance per request: each is an independent one-shot routing test
        result = await build_workflow().run(request)

        # Analyze which agent was activated
        for event in result:
            if event.type != "output":
                continue
            if event.executor_id == "customer_support_agent":
                print(f"Support Agent: {event.data.text[:100]}...")
            elif event.executor_id in agent_labels:
                print(f"Routed to: {agent_labels[event.executor_id]}")
                break

    display(HTML("""
    <div style='padding: 25px; background: linear-gradient(135deg, #9c27b0 0%, #673ab7 100%); color: white; border-radius: 12px; 
                box-shadow: 0 4px 12px rgba(156,39,176,0.4); margin: 20px 0;'>
        <h2 style='margin: 0 0 20px 0;'>Handoff Analysis Results</h2>
        <div style='background: rgba(255,255,255,0.15); padding: 15px; border-radius: 8px;'>
            <h4 style='margin: 0 0 10px 0;'>Key Observations</h4>
            <ul style='margin: 0; padding-left: 20px; line-height: 1.6;'>
                <li><strong>Dynamic Routing:</strong> Customer support agent analyzes request intent</li>
                <li><strong>Context Preservation:</strong> Full conversation history maintained</li>
                <li><strong>Specialist Focus:</strong> Each agent handles their expertise area</li>
                <li><strong>Seamless Handoff:</strong> Users don't need to repeat information</li>
            </ul>
        </div>
    </div>
    """))


# Run the analysis
await analyze_handoff_patterns()

No handoff configuration found for agent 'booking_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'disputes_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'trip_check_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.



--- Test Request 1 ---
User: I want to book a round-trip flight to Tokyo


No handoff configuration found for agent 'booking_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'disputes_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'trip_check_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.


Support Agent: ...
Routed to: 🛫 BOOKING SPECIALIST

--- Test Request 2 ---
User: I need a refund for my cancelled flight


No handoff configuration found for agent 'booking_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'disputes_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'trip_check_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.


Support Agent: ...
Routed to: 💰 DISPUTES SPECIALIST

--- Test Request 3 ---
User: Please check if my travel itinerary is confirmed


No handoff configuration found for agent 'booking_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'disputes_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'trip_check_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.


Support Agent: ...
Routed to: 🎯 TRIP CHECK SPECIALIST

--- Test Request 4 ---
User: Can you help me with a billing dispute?
Support Agent: I'm sorry you're dealing with a billing issue — I can connect you with our disputes team who will he...
Routed to: 💰 DISPUTES SPECIALIST
